# S3 storage unit — end-to-end test (docs/issues/s3_file_engine_support.md)

Exercises the full S3 storage unit lifecycle against a live `mnemosyne_server`
talking to a real S3-compatible endpoint (e.g. MinIO): database creation,
`CREATE STORAGE_UNIT ... TYPE S3`, `CREATE TABLE ... ENGINE=File`, `INSERT`
(twice, to confirm appending rather than overwriting), `SELECT`, and cleanup.

**Credentials:** this notebook reads S3 credentials from environment
variables — it does **not** hardcode a real secret key, so it's safe to keep
in the repo. Before running, set:

```
MNEMOSYNE_TEST_S3_ENDPOINT     e.g. http://100.127.65.29:9000
MNEMOSYNE_TEST_S3_BUCKET       e.g. mnemo
MNEMOSYNE_TEST_S3_ACCESS_KEY
MNEMOSYNE_TEST_S3_SECRET_KEY
```

either in your shell before launching Jupyter, or by editing the "Credentials"
cell below locally (uncommitted) with real values for a quick one-off run.

Run cells top to bottom. The last cell stops the server it started.

In [1]:
import pandas as pd
import sys
sys.path.append("../")
sys.path.append("../scripts")

In [2]:
import json
import os
import subprocess
import time
from pathlib import Path

In [3]:
from _mnemo_client import (
    Client,
    TestRunner,
    find_server_bin,
    GREEN,
    RED,
    YELLOW,
    RESET,
)

## Credentials

Pulled from the environment so nothing secret is committed to this notebook.

In [16]:
S3_ENDPOINT = os.environ.get("MNEMOSYNE_TEST_S3_ENDPOINT", "http://CHANGE_ME:9000")
S3_BUCKET = os.environ.get("MNEMOSYNE_TEST_S3_BUCKET", "CHANGE_ME")
S3_ACCESS_KEY = os.environ.get("MNEMOSYNE_TEST_S3_ACCESS_KEY", "CHANGE_ME")
S3_SECRET_KEY = os.environ.get("MNEMOSYNE_TEST_S3_SECRET_KEY", "CHANGE_ME")

assert "CHANGE_ME" not in (S3_ENDPOINT, S3_BUCKET, S3_ACCESS_KEY, S3_SECRET_KEY), (
    "Set MNEMOSYNE_TEST_S3_ENDPOINT / _BUCKET / _ACCESS_KEY / _SECRET_KEY "
    "(env vars, or edit this cell locally) before running this notebook."
)
print(f"endpoint={S3_ENDPOINT} bucket={S3_BUCKET} access_key={S3_ACCESS_KEY}")

endpoint=http://100.127.65.29:9000 bucket=mnemo access_key=aeacus


## Launch a fresh server

Builds are expected under `build/bin/Debug` or `build/bin/Release` (see
`find_server_bin` in `scripts/_mnemo_client.py`). Rebuild first if this fails
to find a binary:

```
cmake --build build --target mnemosyne_server --config Debug
```

In [17]:
server_bin = find_server_bin()
assert server_bin is not None, (
    "mnemosyne_server binary not found — build it first: "
    "cmake --build build --target mnemosyne_server --config Debug"
)

repo_root = Path("..").resolve()
server_proc = subprocess.Popen(
    [str(server_bin)],
    cwd=str(repo_root),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

client = Client("http://127.0.0.1:1143")

deadline = time.time() + 20
ready = False
while time.time() < deadline:
    if server_proc.poll() is not None:
        raise RuntimeError("mnemosyne_server exited before becoming ready")
    try:
        if client.get("/ping", timeout=1.0).status == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(0.3)

assert ready, "server did not become ready in time"
print(f"{GREEN}Server ready at {client.base_url} (pid={server_proc.pid}, bin={server_bin}){RESET}")

Server ready at http://127.0.0.1:1143 (pid=48240, bin=C:\Users\tsuma.thomas\Documents\Mnemosyne\build\bin\Release\mnemosyne_server.exe)


In [28]:
t = TestRunner()

DB = "s3_verification_db"
UNIT = "s3_verification_unit2"
TABLE = "s3_verification_table2"

t.section("Setup: database")
r = client.query(f"CREATE DATABASE IF NOT EXISTS {DB}")
t.check("CREATE DATABASE", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query(f"USE DATABASE {DB}")
t.check("USE DATABASE", r.ok(), f"status={r.status} body={r.body!r}")


== Setup: database ==
  PASS CREATE DATABASE
  PASS USE DATABASE


True

In [24]:
json.loads(client.query(f"SHOW DATABASES").body)

{'rows': 1,
 'columns': ['databases'],
 'data': [['s3_verification_db']],
 'duration_ms': 0.17,
 'bytes_read': 0,
 'bytes_written': 0}

## `CREATE STORAGE_UNIT ... TYPE S3`

Credentials configured via SQL now actually reach `S3Disk` (previously
silently dropped — see the S3 file engine changelog entry).

In [25]:
json.loads(client.query(f"SHOW STORAGE_UNITS").body)

{'rows': 1,
 'columns': ['name', 'type', 'status'],
 'data': [['s3_verification_unit', 'S3', 'ONLINE']],
 'duration_ms': 0.71,
 'bytes_read': 0,
 'bytes_written': 0}

In [26]:
t.section("CREATE STORAGE_UNIT TYPE S3")

r = client.query(
    f"CREATE STORAGE_UNIT {UNIT} TYPE S3 "
    f"BUCKET '{S3_BUCKET}' ENDPOINT '{S3_ENDPOINT}' "
    f"ACCESS_KEY '{S3_ACCESS_KEY}' SECRET_KEY '{S3_SECRET_KEY}'"
)
t.check("CREATE STORAGE_UNIT TYPE S3", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query("SHOW STORAGE_UNITS")
result = json.loads(r.body)
print(result)
t.check("SHOW STORAGE_UNITS succeeds", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query(f"DESCRIBE STORAGE_UNIT {UNIT}")
result = json.loads(r.body)
print(result)
t.check("DESCRIBE STORAGE_UNIT succeeds", r.ok(), f"status={r.status} body={r.body!r}")

described = {row[0]: row[1] for row in result.get("data", [])}
t.check("DESCRIBE reports type S3", described.get("type") == "S3", f"described={described}")
body_lower = r.body.lower()
t.check(
    "DESCRIBE does not leak access_key/secret_key",
    "access_key" not in body_lower and "secret_key" not in body_lower,
    f"body={r.body!r}",
)


== CREATE STORAGE_UNIT TYPE S3 ==
  PASS CREATE STORAGE_UNIT TYPE S3
{'rows': 2, 'columns': ['name', 'type', 'status'], 'data': [['s3_verification_unit', 'S3', 'ONLINE'], ['s3_verification_unit2', 'S3', 'ONLINE']], 'duration_ms': 0.23, 'bytes_read': 0, 'bytes_written': 0}
  PASS SHOW STORAGE_UNITS succeeds
{'rows': 12, 'columns': ['field', 'value'], 'data': [['name', 's3_verification_unit2'], ['type', 'S3'], ['status', 'ONLINE'], ['path', ''], ['endpoint', 'http://100.127.65.29:9000'], ['bucket', 'mnemo'], ['region', ''], ['capacity_bytes', '0'], ['used_bytes', '0'], ['available_bytes', '0'], ['reference_count', '0'], ['capabilities', 'OBJECT_STORAGE']], 'duration_ms': 0.25, 'bytes_read': 0, 'bytes_written': 0}
  PASS DESCRIBE STORAGE_UNIT succeeds
  PASS DESCRIBE reports type S3
  PASS DESCRIBE does not leak access_key/secret_key


True

## `CREATE TABLE ... ENGINE=File STORAGE_UNIT <s3_unit>`

Previously blocked outright ("S3 storage units do not support File engine I/O
yet"). Now creates a real table backed by S3 objects.

In [29]:
t.section("CREATE TABLE Engine=File on the S3 storage unit")

r = client.query(f"CREATE TABLE {TABLE} (id Float64, val Float64) Engine=File STORAGE_UNIT {UNIT}")
t.check("CREATE TABLE Engine=File STORAGE_UNIT <s3_unit>", r.ok(), f"status={r.status} body={r.body!r}")


== CREATE TABLE Engine=File on the S3 storage unit ==
  PASS CREATE TABLE Engine=File STORAGE_UNIT <s3_unit>


True

## `INSERT` (twice) and `SELECT`

Two separate `INSERT`s confirm rows are appended rather than overwritten —
the same bug fixed for the `LOCAL` storage unit type applies identically here
since `FileStorage` only ever talks to the abstract `IDisk` interface.

In [30]:
t.section("INSERT and SELECT round-trip")

r = client.query(f"INSERT INTO {TABLE} VALUES (1, 1.5), (2, 2.5)")
t.check("first INSERT (2 rows)", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query(f"INSERT INTO {TABLE} VALUES (3, 3.5), (4, 4.5)")
t.check("second INSERT (2 more rows)", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query(f"SELECT * FROM {TABLE}")
result = json.loads(r.body)
print(result)
t.check("query succeeds", r.ok(), f"status={r.status} body={r.body!r}")
t.check(
    "both INSERTs are present (4 rows total, not overwritten)",
    result.get("rows") == 4,
    f"rows={result.get('rows')}",
)


== INSERT and SELECT round-trip ==
  PASS first INSERT (2 rows)
  PASS second INSERT (2 more rows)
{'rows': 4, 'columns': ['id', 'val'], 'data': [[1, 1], [2, 2], [3, 3], [4, 4]], 'duration_ms': 3800.15, 'bytes_read': 0, 'bytes_written': 0}
  PASS query succeeds
  PASS both INSERTs are present (4 rows total, not overwritten)


True

## Verify the objects really landed in S3

Bypasses Mnemosyne entirely and lists the bucket directly with `boto3` (falls
back to a message if `boto3` isn't installed — this check is a nice-to-have,
not required for the SQL-level checks above to be meaningful).

In [31]:
t.section("Direct S3 verification (bypassing Mnemosyne)")

try:
    import boto3
    from botocore.client import Config

    s3 = boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT,
        aws_access_key_id=S3_ACCESS_KEY,
        aws_secret_access_key=S3_SECRET_KEY,
        config=Config(signature_version="s3v4"),
    )
    prefix = f"{DB}/{TABLE}/"
    objects = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix).get("Contents", [])
    sizes = {obj["Key"]: obj["Size"] for obj in objects}
    print(sizes)
    t.check(
        "id.bin and val.bin exist with 32 bytes each (4 rows x 8-byte Float64)",
        sizes.get(prefix + "id.bin") == 32 and sizes.get(prefix + "val.bin") == 32,
        f"sizes={sizes}",
    )
except ImportError:
    print(f"{YELLOW}boto3 not installed — skipping direct S3 verification (pip install boto3){RESET}")


== Direct S3 verification (bypassing Mnemosyne) ==
{'s3_verification_db/s3_verification_table2/id.bin': 32, 's3_verification_db/s3_verification_table2/val.bin': 32}
  PASS id.bin and val.bin exist with 32 bytes each (4 rows x 8-byte Float64)


In [32]:
exit_code = t.summary()


8/8 checks passed


## Cleanup

In [33]:
client.query(f"DROP TABLE {TABLE}")
client.query(f"DROP STORAGE_UNIT {UNIT}")

try:
    import boto3
    from botocore.client import Config

    s3 = boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT,
        aws_access_key_id=S3_ACCESS_KEY,
        aws_secret_access_key=S3_SECRET_KEY,
        config=Config(signature_version="s3v4"),
    )
    prefix = f"{DB}/{TABLE}/"
    objects = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix).get("Contents", [])
    for obj in objects:
        s3.delete_object(Bucket=S3_BUCKET, Key=obj["Key"])
    print(f"Removed {len(objects)} leftover object(s) under {prefix}")
except ImportError:
    pass

server_proc.terminate()
try:
    server_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    server_proc.kill()
print(f"{YELLOW}Server stopped.{RESET}")

Removed 2 leftover object(s) under s3_verification_db/s3_verification_table2/
Server stopped.
